In [ ]:
# Rutas portables: ejecutar desde notebooks/ o desde la raíz clonada.
from pathlib import Path
import sys, os
_repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/pipeline.json').exists())
sys.path.insert(0, str(_repo / 'src'))
from pipeline_runtime import prepare_runtime
_runtime = prepare_runtime(_repo)
os.chdir(_runtime)
sys.path.insert(0, str(_runtime))

import json
_central = json.loads((_repo / 'configs/pipeline.json').read_text(encoding='utf-8'))


# Fase 5.0 — Regresión logística final con WoE

Modelo principal con filtro IV y selección iterativa exclusivamente TRAIN. Se consumen matrices WoE 4.0 existentes; no se calculan nuevos cortes, WoE, IV, PSI ni correlaciones. Ejecutar desde la raíz del proyecto.

Los p-values son inferencia condicionada al modelo: la selección iterativa puede volverlos optimistas. SHAP explica el logit del mismo ajuste statsmodels; sus porcentajes dependen de la muestra/background TRAIN configurados. El umbral 0.50 es una referencia inicial, no un umbral operativo optimizado. Las alertas de deltas son reglas de seguimiento, no pruebas estadísticas.

## MOD.1 — Importaciones y configuración

**Hace:** Importaciones y configuración.

**Recibe:** Módulo 5.0 y parámetros de política.

**Produce:** mod, CONFIG y rutas.

**Funciones:** 5.0.0.modelo_logistico.py: configuration.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: configuration
from pathlib import Path
import importlib.util
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from IPython.display import display
spec = importlib.util.spec_from_file_location('modelo_logistico', '5.0.0.modelo_logistico.py')
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
CONFIG = mod.configuration(dict(
    P_VALUE_THRESHOLD=0.05, SHAP_MIN_PCT=2.0, MAX_IV_MODEL=0.50,
    MAX_ITERACIONES=100, CLASSIFICATION_THRESHOLD=0.50, RANDOM_STATE=42,
    SHAP_SAMPLE_N=2000, SHAP_BACKGROUND_N=500,
    GINI_ALERT_TOLERANCE=0.05, KS_ALERT_TOLERANCE=0.05))
MODEL_ROOT = Path('5.MODELO_LOGISTICO')
pd.set_option('display.max_rows', 150)
pd.set_option('display.max_columns', 30)
display(pd.Series(CONFIG, name='Valor').to_frame())
print('Selección y SHAP exclusivamente TRAIN; sin modelo sklearn alternativo.')
CONFIG = mod.configuration(_central["logistic"])


## MOD.2 — Carga de datos multivariados

**Hace:** Carga de datos multivariados.

**Recibe:** CSV, manifiesto de índices 4.0, targets v4, IV y JSON B.3.

**Produce:** entradas_modelo, matrices, targets y validación.

**Funciones:** 5.0.0.modelo_logistico.py: load_inputs.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: load_inputs
entradas_modelo = mod.load_inputs()
X_train_uncorr, X_test_uncorr, X_oot_uncorr = [entradas_modelo['X'][s] for s in mod.SEGMENTS]
y_train, y_test, y_oot = [entradas_modelo['y'][s] for s in mod.SEGMENTS]
variables_recibidas = entradas_modelo['selection']['retained_variables'].copy()
targets_originales = {s: entradas_modelo['y'][s].copy(deep=True) for s in mod.SEGMENTS}
display(entradas_modelo['validation'])
display(pd.DataFrame({'Variables recibidas de 4.0': variables_recibidas}))

## MOD.3 — Validación de WoE y target

**Hace:** Validación de WoE y target.

**Recibe:** Metadatos B.3, código original y targets.

**Produce:** convencion_woe, signo_esperado.

**Funciones:** 5.0.0.modelo_logistico.py: validate_woe_convention.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: validate_woe_convention
convencion_woe = mod.validate_woe_convention(entradas_modelo['artifact'],
    Path('3.0.0.bivariado.py').read_text(encoding='utf-8-sig'))
signo_esperado = convencion_woe['expected_sign']
display(pd.Series(convencion_woe, name='Validación').to_frame())
print('Target comprobado:', y_train.name, '| 1 = default/malo, 0 = bueno')
print('WoE ln(%Buenos/%Malos): coeficiente esperado negativo; intercepto exento.')
print('Fórmula verificada en AST y JSON; no se recalculan WoE.')

## MOD.4 — Ranking y exclusión por IV > 0.50

**Hace:** Ranking y exclusión por IV > 0.50.

**Recibe:** IV final B.3 validado por 4.0 y variables recibidas.

**Produce:** variables_iniciales, df_filtro_iv, X_train_modelo_inicial.

**Funciones:** 5.0.0.modelo_logistico.py: filter_iv.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: filter_iv
variables_iniciales, df_filtro_iv = mod.filter_iv(entradas_modelo['iv'], variables_recibidas, CONFIG['MAX_IV_MODEL'])
X_train_modelo_inicial = X_train_uncorr.loc[:, variables_iniciales].copy()
display(df_filtro_iv)
print('IV alto: alerta de revisión por política; no prueba automática de fuga.')

## MOD.5 — Modelo logístico inicial

**Hace:** Modelo logístico inicial.

**Recibe:** TRAIN tras filtro IV, y_train, signo y CONFIG.

**Produce:** estado_seleccion y modelo_inicial_sm.

**Funciones:** 5.0.0.modelo_logistico.py: SeleccionIterativa, fit_logit, coefficient_table.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: SeleccionIterativa, fit_logit, coefficient_table
estado_seleccion = mod.SeleccionIterativa(X_train_modelo_inicial, y_train, signo_esperado, CONFIG)
modelo_inicial_sm = estado_seleccion.model
display(mod.coefficient_table(modelo_inicial_sm))
display(estado_seleccion.df_iteraciones_modelo)
print('Ajuste inicial convergente y estabilidad numérica comprobada.')

## MOD.6 — Iteraciones por signo

**Hace:** Iteraciones por signo.

**Recibe:** Estado inicial y TRAIN.

**Produce:** Estado sin coeficientes de signo contrario.

**Funciones:** 5.0.0.modelo_logistico.py: eliminar_variables_completo(etapa=signo).

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: eliminar_variables_completo(etapa=signo)
estado_seleccion = mod.eliminar_variables_completo(X_train_modelo_inicial, y_train, signo_esperado,
    CONFIG, estado=estado_seleccion, etapa='signo')
display(estado_seleccion.df_iteraciones_modelo)
print('Regla: mayor magnitud de signo contrario, empate alfabético; un descarte y un reajuste por vez.')

## MOD.7 — Iteraciones por p-value

**Hace:** Iteraciones por p-value.

**Recibe:** Estado sin signos contrarios y TRAIN.

**Produce:** Estado con p-values dentro de política.

**Funciones:** 5.0.0.modelo_logistico.py: eliminar_variables_completo(etapa=pvalue).

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: eliminar_variables_completo(etapa=pvalue)
estado_seleccion = mod.eliminar_variables_completo(X_train_modelo_inicial, y_train, signo_esperado,
    CONFIG, estado=estado_seleccion, etapa='pvalue')
display(estado_seleccion.df_iteraciones_modelo)
display(mod.coefficient_table(estado_seleccion.model))
print('Mayor p-value > umbral; tras cada descarte se revisan también signos.')
print('Inferencia condicionada; selección iterativa puede hacer optimistas los p-values.')

## MOD.8 — Iteraciones SHAP

**Hace:** Iteraciones SHAP.

**Recibe:** Logit vigente, TRAIN y muestra/background reproducibles.

**Produce:** Modelo validado y SHAP del mismo ajuste.

**Funciones:** 5.0.0.modelo_logistico.py: eliminar_variables_completo(etapa=shap), linear_shap.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: eliminar_variables_completo(etapa=shap), linear_shap
estado_seleccion = mod.eliminar_variables_completo(X_train_modelo_inicial, y_train, signo_esperado,
    CONFIG, estado=estado_seleccion, etapa='shap')
assert estado_seleccion.status == 'VALIDADO'
display(estado_seleccion.shap['table'])
print('SHAP interventional en log-odds; se comprueba aditividad con el Logit vigente.')
print('Tras eliminar por SHAP se vuelven a evaluar signos y p-values.')

## MOD.9 — Historial de eliminaciones

**Hace:** Historial de eliminaciones.

**Recibe:** Todos los modelos evaluados.

**Produce:** df_iteraciones_modelo, df_coeficientes_iteraciones y figura de evolución.

**Funciones:** 5.0.0.modelo_logistico.py: plot_iterations.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: plot_iterations
df_iteraciones_modelo = estado_seleccion.df_iteraciones_modelo
df_coeficientes_iteraciones = estado_seleccion.df_coeficientes_iteraciones
display(df_iteraciones_modelo)
display(df_coeficientes_iteraciones)
figuras_modelo = {}
figuras_modelo['evolucion_variables'] = mod.plot_iterations(df_iteraciones_modelo)
plt.show()

## MOD.10 — Modelo final y summary() de statsmodels

**Hace:** Modelo final y summary() de statsmodels.

**Recibe:** Resultado Logit ya ajustado y validado.

**Produce:** modelo_sm, variables_finales_modelo y tabla de coeficientes.

**Funciones:** statsmodels: LogitResults.summary; 5.0.0.modelo_logistico.py: coefficient_table, plot_coefficients.

In [ ]:
# Funciones: statsmodels: LogitResults.summary; 5.0.0.modelo_logistico.py: coefficient_table, plot_coefficients
modelo_sm = estado_seleccion.model
variables_finales_modelo = estado_seleccion.variables.copy()
X_train_modelo = X_train_uncorr.loc[:, variables_finales_modelo].copy()
# Reutilizar el MISMO ajuste seleccionado: no efectuar un fit adicional.
print(modelo_sm.summary())
df_coeficientes_final = mod.coefficient_table(modelo_sm)
display(df_coeficientes_final)
figuras_modelo['coeficientes_ic'] = mod.plot_coefficients(modelo_sm)
plt.show()

## MOD.11 — Predicciones TRAIN/TEST/OOT

**Hace:** Predicciones TRAIN/TEST/OOT.

**Recibe:** Mismo Logit TRAIN y columnas finales ordenadas.

**Produce:** matrices_modelo y y_proba_train/test/oot.

**Funciones:** 5.0.0.modelo_logistico.py: predict_segments.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: predict_segments
matrices_modelo = {s: entradas_modelo['X'][s].loc[:, variables_finales_modelo].copy() for s in mod.SEGMENTS}
X_test_modelo, X_oot_modelo = matrices_modelo['TEST'], matrices_modelo['OOT']
predicciones_modelo = mod.predict_segments(modelo_sm, matrices_modelo, variables_finales_modelo)
y_proba_train, y_proba_test, y_proba_oot = [predicciones_modelo[s] for s in mod.SEGMENTS]
for segmento in mod.SEGMENTS:
    assert matrices_modelo[segmento].columns.tolist() == variables_finales_modelo
    assert predicciones_modelo[segmento].index.equals(entradas_modelo['y'][segmento].index)
    pd.testing.assert_series_equal(targets_originales[segmento], entradas_modelo['y'][segmento], check_exact=True)
    print(segmento, matrices_modelo[segmento].shape, '| probabilidades finitas en [0,1]')

## MOD.12 — AUC, Gini y KS

**Hace:** AUC, Gini y KS.

**Recibe:** Targets y predicciones por segmento.

**Produce:** df_metricas_modelo consolidado y métricas de discriminación.

**Funciones:** 5.0.0.modelo_logistico.py: calcular_metricas, gini_auc, ks_stat.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: calcular_metricas, gini_auc, ks_stat
df_metricas_modelo = pd.DataFrame([dict(Segmento=s, **mod.calcular_metricas(
    entradas_modelo['y'][s], predicciones_modelo[s], CONFIG['CLASSIFICATION_THRESHOLD'])) for s in mod.SEGMENTS])
display(df_metricas_modelo[['Segmento','N','AUC','Gini','KS']])

## MOD.13 — Recall, Precision, F1, Brier y Accuracy

**Hace:** Recall, Precision, F1, Brier y Accuracy.

**Recibe:** df_metricas_modelo y umbral de referencia.

**Produce:** Tabla de clasificación, calibración y advertencias.

**Funciones:** 5.0.0.modelo_logistico.py: resultados de calcular_metricas.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: resultados de calcular_metricas
display(df_metricas_modelo[['Segmento','Recall','Precision','F1_score','Brier','Accuracy',
    'Tasa_default_observada','Probabilidad_promedio','TN','FP','FN','TP','Umbral','Advertencias']])
print('Umbral 0.50 de referencia; no optimizado en TEST/OOT.')
print('Las métricas indefinidas se registran NaN con su causa.')

## MOD.14 — Comparación TRAIN/TEST/OOT y deltas

**Hace:** Comparación TRAIN/TEST/OOT y deltas.

**Recibe:** Métricas y tolerancias configuradas antes del ajuste.

**Produce:** Bloques de métricas y df_deltas_modelo.

**Funciones:** 5.0.0.modelo_logistico.py: print_metricas, segment_deltas.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: print_metricas, segment_deltas
mod.print_metricas(df_metricas_modelo)
df_deltas_modelo = mod.segment_deltas(df_metricas_modelo, CONFIG)
display(df_deltas_modelo)
print('Alertas según |delta| y tolerancia configurada: seguimiento descriptivo, no prueba estadística.')

## MOD.15 — Curvas ROC y KS

**Hace:** Curvas ROC y KS.

**Recibe:** Targets y predicciones ya calculadas.

**Produce:** figuras ROC y KS en memoria.

**Funciones:** 5.0.0.modelo_logistico.py: plot_roc, plot_ks.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: plot_roc, plot_ks
figuras_modelo['roc_segmentos'] = mod.plot_roc(entradas_modelo['y'], predicciones_modelo)
plt.show()

In [ ]:
# MOD.15B — KS independiente. Función: 5.0.0.modelo_logistico.py: plot_ks
figuras_modelo['ks_segmentos'] = mod.plot_ks(entradas_modelo['y'], predicciones_modelo)
plt.show()

## MOD.16 — Matrices de confusión

**Hace:** Matrices de confusión.

**Recibe:** Métricas al umbral configurado.

**Produce:** Figura de matrices por segmento.

**Funciones:** 5.0.0.modelo_logistico.py: plot_confusion.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: plot_confusion
figuras_modelo['matrices_confusion'] = mod.plot_confusion(df_metricas_modelo)
plt.show()

## MOD.17 — Calibración

**Hace:** Calibración.

**Recibe:** Targets y probabilidades reales.

**Produce:** Figura de calibración por segmento.

**Funciones:** 5.0.0.modelo_logistico.py: plot_calibration.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: plot_calibration
figuras_modelo['calibracion'] = mod.plot_calibration(entradas_modelo['y'], predicciones_modelo)
plt.show()

## MOD.18 — SHAP final

**Hace:** SHAP final.

**Recibe:** SHAP del Logit seleccionado, calculado sobre TRAIN.

**Produce:** Ranking SHAP final y figura.

**Funciones:** 5.0.0.modelo_logistico.py: plot_shap.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: plot_shap
df_shap_final = estado_seleccion.shap['table']
display(df_shap_final)
print('Muestra:', len(estado_seleccion.shap['values']), '| Background:',
    len(estado_seleccion.shap['background_indices']), '| seed:', CONFIG['RANDOM_STATE'])
figuras_modelo['shap_final'] = mod.plot_shap(estado_seleccion.shap)
plt.show()

## MOD.19 — Guardar modelo y artefactos

**Hace:** Guardar modelo y artefactos.

**Recibe:** Estado validado, matrices, métricas, figuras y hashes.

**Produce:** Modelo serializado, JSON de coeficientes/configuración y CSV.

**Funciones:** 5.0.0.modelo_logistico.py: export_results.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: export_results
configuracion_modelo = mod.export_results(MODEL_ROOT, estado_seleccion, entradas_modelo,
    matrices_modelo, predicciones_modelo, df_metricas_modelo, df_deltas_modelo,
    df_filtro_iv, convencion_woe, figuras_modelo)
entradas_modelo['validation'].to_csv(MODEL_ROOT / 'evidencias/validacion_entradas.csv', index=False)
print('Modelo recuperado: probabilidades reproducidas con rtol=atol=1e-12 en las tres muestras.')
print('Artefactos guardados en:', MODEL_ROOT)

## MOD.20 — Resumen final

**Hace:** Resumen final.

**Recibe:** Resultados reales de selección y evaluación.

**Produce:** Resumen de exclusiones, variables finales, métricas y pruebas.

**Funciones:** 5.0.0.modelo_logistico.py: print_metricas.

In [ ]:
# Funciones: 5.0.0.modelo_logistico.py: print_metricas
print('Variables recibidas de correlación:', len(variables_recibidas))
print('Excluidas por IV:', df_filtro_iv.loc[df_filtro_iv.Estado_IV.eq('EXCLUIDA_IV'),'Variable'].tolist())
for criterio in ['SIGNO','P_VALUE','SHAP']:
    print('Eliminadas por', criterio, ':', df_iteraciones_modelo.loc[
        df_iteraciones_modelo.Criterio_eliminacion.eq(criterio),'Variable_eliminada'].tolist())
print('Variables finales:', len(variables_finales_modelo))
display(pd.DataFrame({'Variable':variables_finales_modelo}))
mod.print_metricas(df_metricas_modelo)
display(df_deltas_modelo)
pruebas_path = MODEL_ROOT / 'evidencias/pruebas.json'
if pruebas_path.exists(): display(pd.DataFrame([json.loads(pruebas_path.read_text(encoding='utf-8'))]))
print('TRAIN es rendimiento de desarrollo; TEST y OOT no participaron en selección ni ajuste.')